In [0]:
dbutils.widgets.text("p_batch_id", "", "Batch ID")

v_batch_id = dbutils.widgets.get("p_batch_id").strip()

if not v_batch_id:
    raise ValueError("Enter a batch ID")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
bronze_path = "s3://bizf1project123/Bronze/Aaisha/incremental-load/circuits"

silver_path = "s3://bizf1project123/Silver/Aaisha/incremental-load/circuits"

In [0]:
circuits_df = (
    spark.read
        .format("delta")
        .load(bronze_path)
        .filter(F.col("batch_id") == v_batch_id)
)

In [0]:
display(circuits_df)

In [0]:
circuits_selected_df = circuits_df.select(
    "circuitId",
    "circuitName",
    "lat",
    "long",
    "locality",
    "country",
    "ingestion_timestamp",
    "source_file",
    "batch_id"
)

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnRenamed("circuitId", "circuit_id")
        .withColumnRenamed("circuitName", "circuit_name")
        .withColumnRenamed("lat", "latitude")
        .withColumnRenamed("long", "longitude")
)

In [0]:
display(circuits_renamed_df)

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(
    ["circuit_id"]
)

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn(
            "circuit_name",
            F.initcap(F.col("circuit_name"))
        )
        .withColumn(
            "locality",
            F.initcap(F.col("locality"))
        )
        .withColumn(
            "created_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "updated_timestamp",
            F.current_timestamp()
        )
)

In [0]:
display(circuits_final_df)

In [0]:
if not DeltaTable.isDeltaTable(spark, silver_path):

    (
        circuits_final_df.write
            .format("delta")
            .mode("overwrite")
            .save(silver_path)
    )

else:

    silver_delta = DeltaTable.forPath(spark, silver_path)

    (
        silver_delta.alias("t")
            .merge(
                circuits_final_df.alias("s"),
                "t.circuit_id = s.circuit_id"
            )
            .whenMatchedUpdate(
                condition="s.batch_id >= t.batch_id",
                set={
                    "circuit_name": "s.circuit_name",
                    "latitude": "s.latitude",
                    "longitude": "s.longitude",
                    "locality": "s.locality",
                    "country": "s.country",
                    "ingestion_timestamp": "s.ingestion_timestamp",
                    "source_file": "s.source_file",
                    "batch_id": "s.batch_id",
                    "updated_timestamp": "s.updated_timestamp"
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
    )

In [0]:
circuits_silver_df = (
    spark.read
        .format("delta")
        .load(silver_path)
)

display(circuits_silver_df)